# 3- Trying the models

Classifier: SVM (linear), RF

In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import cognitive_models.preprocessing as cwpre
import cognitive_models.features as cwfeat
import cognitive_models.interpolate as cwinterp
import cognitive_models.pupil_utils as cwpupil
from importlib import reload
reload(cwpupil)
reload(cwinterp)
reload(cwpre)
reload(cwfeat)

COLET_DATASET_DIR = Path(cwpre.__file__).parents[2] / "datasets" / "COLET_CSV"
PARTICIPANTS = list(range(1, 11)) # Participants 1 to 10
PARTICIPANTS.remove(6) # Bad data
EXPERIMENTS = [1,2,3,4]

### A. Getting all the needed features

In [49]:
# 1- Load all required data
all_eye_data = cwpre.load_colet_data(COLET_DATASET_DIR, PARTICIPANTS, EXPERIMENTS)

print(f"Loaded eye-tracking data for {len(PARTICIPANTS)} participants and {len(EXPERIMENTS)} experiments. Total records: {len(all_eye_data)}")

There are 1 non matching timestamp_sec within the limits
Final merged dataset has 8184 records at 240 Hz
Final merged and resampled dataset has 2028 records at 60 Hz
There are 1 non matching timestamp_sec within the limits
Final merged dataset has 6866 records at 240 Hz
Final merged and resampled dataset has 1709 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 17301 records at 240 Hz
Final merged and resampled dataset has 4285 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 9246 records at 240 Hz
Final merged and resampled dataset has 2290 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 10159 records at 240 Hz
Final merged and resampled dataset has 2505 records at 60 Hz
There are 0 non matching timestamp_sec within the limits
Final merged dataset has 7113 records at 240 Hz
Final merged and resampled dataset has 1769 records at 60 Hz
Th

In [50]:
# Lets check integrity of the data
all_eye_data.columns
all_eye_data.groupby(['subject_id', 'task_id'])['cl_class'].value_counts().sort_index().head()

subject_id  task_id  cl_class
1           1        low         2028
            2        medium      1709
            3        high        4285
            4        medium      2290
2           1        low         2505
Name: count, dtype: int64

In [63]:
import tqdm
all_eye_data_grouped = all_eye_data.groupby(['subject_id', 'task_id'])
# Extract all window features
Fs = 60
WINDOW_N = 512
WINDOW_STEP = 64
SKIP_EDGE_SAMPLES = Fs # first & last 2 seconds are discarded

features_df = pd.DataFrame()
for (subject_id, task_id), group_df in tqdm.tqdm(all_eye_data_grouped, desc="Extracting features"):
    feature_rows = []
    # Discard edge samples
    group_df = group_df.iloc[SKIP_EDGE_SAMPLES:-SKIP_EDGE_SAMPLES].sort_values('timestamp_sec')
    
    # Extract features in sliding windows
    for start in range(WINDOW_N, len(group_df), WINDOW_STEP):
        window_df = group_df.iloc[start - WINDOW_N:start].reset_index(drop=True)
        window_df, blink_df = cwpre.preprocess_colet_data(window_df, verbose=False) # Add derived columns
        features = cwfeat.extract_window_features(window_df, blink_df, ivt_threshold=45, min_fixation_duration=55, verbose=False)
        features['subject_id'] = subject_id
        features['task_id'] = task_id
        features['cl_class'] = window_df['cl_class'].iloc[0] # Class is the same for a given task
        feature_rows.append(features)
    
    features_df = pd.concat([features_df, pd.DataFrame(feature_rows)], ignore_index=True)

print(f"Extracted features for {len(features_df)} windows.")

Extracting features:  53%|█████▎    | 19/36 [01:21<01:16,  4.48s/it]C:\Users\ahebert\Desktop\user-adaptive-swarm-interfaces\research\cognitive_load_models\src\cognitive_models\pupil_utils.py:27: RuntimeWarning: divide by zero encountered in divide
  cD_LH = cD_L / cD_H[[i for i in range(len(cD_H)) if i % (2 ** (lof - hif)) == 0]]
C:\Users\ahebert\Desktop\user-adaptive-swarm-interfaces\research\cognitive_load_models\src\cognitive_models\pupil_utils.py:47: RuntimeWarning: invalid value encountered in multiply
  return np.multiply(
C:\Users\ahebert\Desktop\user-adaptive-swarm-interfaces\research\cognitive_load_models\src\cognitive_models\pupil_utils.py:27: RuntimeWarning: divide by zero encountered in divide
  cD_LH = cD_L / cD_H[[i for i in range(len(cD_H)) if i % (2 ** (lof - hif)) == 0]]
C:\Users\ahebert\Desktop\user-adaptive-swarm-interfaces\research\cognitive_load_models\src\cognitive_models\pupil_utils.py:27: RuntimeWarning: invalid value encountered in divide
  cD_LH = cD_L / cD_H[

Extracted features for 1213 windows.


In [64]:
# Check what is the issue with the NaNs
features_df[features_df.isna().any(axis=1)].head()

,fixations_count,fixations_duration_mean,fixations_duration_max,fixations_duration_min,fixations_duration_std,saccades_count,saccades_peak_velocity_mean,saccades_amplitude_mean,saccades_amplitude_max,saccades_amplitude_min,...,saccades_duration_mean,saccades_duration_max,saccades_duration_min,saccades_duration_std,blinks_count,blinks_duration_mean,pupil_lhipa,subject_id,task_id,cl_class


In [65]:
from sklearn import preprocessing
# z-normalize features
features_df_transformed = features_df.copy()
feature_cols = [col for col in features_df.columns if col not in ['subject_id', 'task_id', 'cl_class']]
scaler = preprocessing.StandardScaler()
features_df_transformed[feature_cols] = scaler.fit_transform(features_df[feature_cols])

features_df_transformed.describe()

,fixations_count,fixations_duration_mean,fixations_duration_max,fixations_duration_min,fixations_duration_std,saccades_count,saccades_peak_velocity_mean,saccades_amplitude_mean,saccades_amplitude_max,saccades_amplitude_min,saccades_amplitude_std,saccades_duration_mean,saccades_duration_max,saccades_duration_min,saccades_duration_std,blinks_count,blinks_duration_mean,pupil_lhipa,subject_id,task_id
count,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1.213000e+03,1213.000000,1213.000000
mean,1.347278e-16,7.029277e-17,1.874474e-16,6.443504e-17,2.343092e-17,-3.221752e-17,-3.397484e-16,4.100412e-17,1.171546e-17,1.171546e-17,1.757319e-17,2.343092e-16,3.807525e-17,5.564844e-16,7.615050e-17,4.686185e-17,4.686185e-17,1.540583e-15,5.421270,2.834295
std,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,1.000412e+00,2.922787,1.049003
min,-2.517652e+00,-1.848083e+00,-1.775078e+00,-6.401348e-01,-1.862207e+00,-2.840228e+00,-1.348575e+00,-6.393607e-01,-3.919661e-01,-4.778897e-01,-3.824018e-01,-9.952350e-01,-8.353907e-01,-4.103802e+00,-7.667418e-01,-1.008002e+00,-1.628952e+00,-3.952718e+00,1.000000,1.000000
25%,-8.377299e-01,-6.594514e-01,-6.895025e-01,-6.401348e-01,-6.915586e-01,-6.540508e-01,-7.405330e-01,-3.582572e-01,-3.463808e-01,-4.468715e-01,-3.345149e-01,-5.778775e-01,-6.730849e-01,6.879802e-02,-6.005601e-01,-6.637714e-01,-1.634101e-01,-8.052474e-01,3.000000,2.000000
50%,9.556030e-02,-2.166279e-01,-2.441381e-01,-2.870273e-01,-2.574433e-01,1.802289e-03,-2.530360e-01,-2.809310e-01,-3.340491e-01,-2.987133e-01,-3.211463e-01,-3.419798e-01,-4.025753e-01,6.879802e-02,-4.195892e-01,-3.195411e-01,3.251038e-01,-1.058094e-01,5.000000,3.000000
75%,6.555345e-01,3.627053e-01,5.074143e-01,6.608031e-02,3.942390e-01,6.576554e-01,4.148888e-01,-1.587641e-01,-2.536201e-01,9.383480e-02,-2.496566e-01,2.194782e-01,2.466476e-01,6.879802e-02,2.104937e-01,3.689196e-01,6.914892e-01,5.936285e-01,8.000000,4.000000
max,2.708773e+00,7.359096e+00,4.376517e+00,1.101241e+01,4.624591e+00,2.625215e+00,6.793464e+00,9.675574e+00,5.917424e+00,1.685772e+01,5.981732e+00,1.052746e+01,4.574801e+00,8.413998e+00,6.508501e+00,4.499684e+00,2.279159e+00,3.391380e+00,10.000000,4.000000


In [66]:
from sklearn.model_selection import train_test_split
from sklearn import svm

features_df['labels'] = features_df['cl_class'].map({'low': 0, 'medium': 1, 'high': 2})

X = features_df[feature_cols].values
y = features_df['labels'].values

train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_x)}, Test set size: {len(test_x)}")

model = svm.LinearSVC()
model.fit(train_x, train_y)
train_acc = model.score(train_x, train_y)
test_acc = model.score(test_x, test_y)
print(f"Train Accuracy: {train_acc:.2f}, Test Accuracy: {test_acc:.2f}")

Training set size: 970, Test set size: 243
Train Accuracy: 0.61, Test Accuracy: 0.60
